## SRP220133 / GSE136774

**paper:** [PMID: 31668620](https://pmc.ncbi.nlm.nih.gov/articles/PMC6834345/) - Attenuated Fgf signaling underlies the forelimb heterochrony in the emu Dromaius novaehollandiae, 2020

**date, curator:** 2026-03-10, Sara Carsanaro

**notes**
* per methods, some libraries are TruSeq Stranded mRNA and some are KAPA RNA HyperPrep Kit
* i did stage annotations manually because some need chicken species-specific developmental ontology and some need general (emu)
    * all HH stages are part of organogenesis stage, so that is the stage i used for the emu
* chicken strain is White leghorn

### annotation summary

In [20]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Flank somatopleure,UBERON:0004874,somatopleure,missing child term
3,Hindlimb somatopleure,UBERON:0004874,somatopleure,missing child term
7,Forelimb somatopleure,UBERON:0004874,somatopleure,missing child term
11,Splanchnopleure at the forelimb level,UBERON:0004874,somatopleure,missing child term
13,Somatorpleure at the forelimb leve,UBERON:0004874,somatopleure,missing child term
16,Undifferentiated lateral plate mesoderm at the forelimb level,UBERON:0005733,limb field,missing child term


In [21]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,HH18,UBERON:0000111,organogenesis stage,other
11,HH13,UBERON:0000111,organogenesis stage,other
15,HH10,UBERON:0000111,organogenesis stage,other
19,HH13,GgalDv:0000023,Hamburger Hamilton stage 13,perfect match
22,HH10,GgalDv:0000020,Hamburger Hamilton stage 10,perfect match
25,HH18,GgalDv:0000030,Hamburger Hamilton stage 18,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP220133"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
0it [00:00, ?it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes
Fetching RunIds and ReadHashes for each library...can take several minutes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX6793622,SRP220133,NextSeq 500,SRS5335555,,,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057700,Flank somatopleure,HH18,,,other,,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,707K,"SAMN12684357,GSM4057700",,,,,,,,,,,10/03/2026,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059715,0C7656C2B9AD9276097F28C2272F0692
1,SRX6793621,SRP220133,NextSeq 500,SRS5335554,,,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057699,Flank somatopleure,HH18,,,other,,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,595K,"SAMN12684360,GSM4057699",,,,,,,,,,,10/03/2026,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059714,D46FBDB4F4093A88AD4442A41AFAFCCE
2,SRX6793620,SRP220133,NextSeq 500,SRS5335553,,,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057698,Flank somatopleure,HH18,,,other,,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,583K,"SAMN12684361,GSM4057698",,,,,,,,,,,10/03/2026,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059713,B18481D02FAC8DCE753FDE79BF07BA5D
3,SRX6793619,SRP220133,NextSeq 500,SRS5335552,,,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057697,Hindlimb somatopleure,HH18,,,other,,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,710HL,"SAMN12684345,GSM4057697",,,,,,,,,,,10/03/2026,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059712,1B57FA31B14B8B642C9E27A2C2001877
4,SRX6793618,SRP220133,NextSeq 500,SRS5335551,,,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057696,Hindlimb somatopleure,HH18,,,other,,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,707HL,"SAMN12684346,GSM4057696",,,,,,,,,,,10/03/2026,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059711,F1977DDB98252A0DFE448A807A193AED
5,SRX6793617,SRP220133,NextSeq 500,SRS5335550,,,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057695,Hindlimb somatopleure,HH18,,,other,,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,595HL,"SAMN12684347,GSM4057695",,,,,,,,,,,10/03/2026,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of ampl

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['Flank somatopleure' 'Forelimb somatopleure' 'Hindlimb somatopleure'
 'Somatorpleure at the forelimb leve'
 'Splanchnopleure at the forelimb level'
 'Undifferentiated lateral plate mesoderm at the forelimb level']


In [7]:

# Flank somatopleure
library.loc[library["infoOrgan"] == "Flank somatopleure", "anatId"] = "UBERON:0004874"
library.loc[library["infoOrgan"] == "Flank somatopleure", "anatName"] = "somatopleure"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Flank somatopleure", "anatAnnotationStatus"] = "missing child term"

# Forelimb somatopleure
library.loc[library["infoOrgan"] == "Forelimb somatopleure", "anatId"] = "UBERON:0004874"
library.loc[library["infoOrgan"] == "Forelimb somatopleure", "anatName"] = "somatopleure"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Forelimb somatopleure", "anatAnnotationStatus"] = "missing child term"

# Hindlimb somatopleure
library.loc[library["infoOrgan"] == "Hindlimb somatopleure", "anatId"] = "UBERON:0004874"
library.loc[library["infoOrgan"] == "Hindlimb somatopleure", "anatName"] = "somatopleure"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Hindlimb somatopleure", "anatAnnotationStatus"] = "missing child term"

# Somatorpleure at the forelimb leve
library.loc[library["infoOrgan"] == "Somatorpleure at the forelimb leve", "anatId"] = "UBERON:0004874"
library.loc[library["infoOrgan"] == "Somatorpleure at the forelimb leve", "anatName"] = "somatopleure"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Somatorpleure at the forelimb leve", "anatAnnotationStatus"] = "missing child term"

# Splanchnopleure at the forelimb level
library.loc[library["infoOrgan"] == "Splanchnopleure at the forelimb level", "anatId"] = "UBERON:0004874"
library.loc[library["infoOrgan"] == "Splanchnopleure at the forelimb level", "anatName"] = "somatopleure"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Splanchnopleure at the forelimb level", "anatAnnotationStatus"] = "missing child term"

# Undifferentiated lateral plate mesoderm at the forelimb level
library.loc[library["infoOrgan"] == "Undifferentiated lateral plate mesoderm at the forelimb level", "anatId"] = "UBERON:0005733"
library.loc[library["infoOrgan"] == "Undifferentiated lateral plate mesoderm at the forelimb level", "anatName"] = "limb field"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Undifferentiated lateral plate mesoderm at the forelimb level", "anatAnnotationStatus"] = "missing child term"


# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX6793622,SRP220133,NextSeq 500,SRS5335555,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057700,Flank somatopleure,HH18,missing child term,not documented,other,,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,707K,"SAMN12684357,GSM4057700",,,,,,,,,,,10/03/2026,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059715,0C7656C2B9AD9276097F28C2272F0692
1,SRX6793621,SRP220133,NextSeq 500,SRS5335554,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057699,Flank somatopleure,HH18,missing child term,not documented,other,,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,595K,"SAMN12684360,GSM4057699",,,,,,,,,,,10/03/2026,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059714,D46FBDB4F4093A88AD4442A41AFAFCCE
2,SRX6793620,SRP220133,NextSeq 500,SRS5335553,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057698,Flank somatopleure,HH18,missing child term,not documented,other,,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,583K,"SAMN12684361,GSM4057698",,,,,,,,,,,10/03/2026,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059713,B18481D02FAC8DCE753FDE79BF07BA5D
3,SRX6793619,SRP220133,NextSeq 500,SRS5335552,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057697,Hindlimb somatopleure,HH18,missing child term,not documented,other,,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,710HL,"SAMN12684345,GSM4057697",,,,,,,,,,,10/03/2026,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059712,1B57FA31B14B8B642C9E27A2C2001877
4,SRX6793618,SRP220133,NextSeq 500,SRS5335551,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057696,Hindlimb somatopleure,HH18,missing child term,not documented,other,,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,707HL,"SAMN12684346,GSM4057696",,,,,,,,,,,10/03/2026,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059711,F1977DDB98252A0DFE448A807A193AED
5,SRX6793617,SRP220133,NextSeq 500,SRS5335550,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057695,Hindlimb somatopleure,HH18,

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [6]:
unique_sorted(library, "infoStage")

['HH10' 'HH13' 'HH18']


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [8]:
library.loc[:,'sex'] = 'NA'
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX6793622,SRP220133,NextSeq 500,SRS5335555,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057700,Flank somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,707K,"SAMN12684357,GSM4057700",,,,,,,,,,,10/03/2026,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059715,0C7656C2B9AD9276097F28C2272F0692
1,SRX6793621,SRP220133,NextSeq 500,SRS5335554,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057699,Flank somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,595K,"SAMN12684360,GSM4057699",,,,,,,,,,,10/03/2026,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059714,D46FBDB4F4093A88AD4442A41AFAFCCE
2,SRX6793620,SRP220133,NextSeq 500,SRS5335553,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057698,Flank somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,583K,"SAMN12684361,GSM4057698",,,,,,,,,,,10/03/2026,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059713,B18481D02FAC8DCE753FDE79BF07BA5D
3,SRX6793619,SRP220133,NextSeq 500,SRS5335552,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057697,Hindlimb somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,710HL,"SAMN12684345,GSM4057697",,,,,,,,,,,10/03/2026,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059712,1B57FA31B14B8B642C9E27A2C2001877
4,SRX6793618,SRP220133,NextSeq 500,SRS5335551,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057696,Hindlimb somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,707HL,"SAMN12684346,GSM4057696",,,,,,,,,,,10/03/2026,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059711,F1977DDB98252A0DFE448A807A193AED
5,SRX6793617,SRP220133,NextSeq 500,SRS5335550,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057695,Hindlimb somatopl

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
my_protocol = 'TruSeq Stranded mRNA, KAPA mRNA HyperPrep Kit'
# full_length or 3'
my_protocolType = 'full_length'

#library.loc[:,'protocol'] = my_protocol
#library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
#library.loc[:,'RNASelection'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX6793622,SRP220133,NextSeq 500,SRS5335555,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057700,Flank somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,707K,"SAMN12684357,GSM4057700",,,,,,,,,,,10/03/2026,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059715,0C7656C2B9AD9276097F28C2272F0692
1,SRX6793621,SRP220133,NextSeq 500,SRS5335554,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057699,Flank somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,595K,"SAMN12684360,GSM4057699",,,,,,,,,,,10/03/2026,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059714,D46FBDB4F4093A88AD4442A41AFAFCCE
2,SRX6793620,SRP220133,NextSeq 500,SRS5335553,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057698,Flank somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,583K,"SAMN12684361,GSM4057698",,,,,,,,,,,10/03/2026,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059713,B18481D02FAC8DCE753FDE79BF07BA5D
3,SRX6793619,SRP220133,NextSeq 500,SRS5335552,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057697,Hindlimb somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,710HL,"SAMN12684345,GSM4057697",,,,,,,,,,,10/03/2026,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059712,1B57FA31B14B8B642C9E27A2C2001877
4,SRX6793618,SRP220133,NextSeq 500,SRS5335551,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057696,Hindlimb somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,707HL,"SAMN12684346,GSM4057696",,,,,,,,,,,10/03/2026,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059711,F1977DDB98252A0DFE448A807A193AED
5,SRX6793617,SRP220133,NextSeq 500,SRS5335550,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057695,Hindlimb somatopl

#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [11]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-03-10'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX6793622,SRP220133,NextSeq 500,SRS5335555,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057700,Flank somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,707K,"SAMN12684357,GSM4057700",,,,,,,,,,SAC,2026-03-10,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059715,0C7656C2B9AD9276097F28C2272F0692
1,SRX6793621,SRP220133,NextSeq 500,SRS5335554,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057699,Flank somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,595K,"SAMN12684360,GSM4057699",,,,,,,,,,SAC,2026-03-10,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059714,D46FBDB4F4093A88AD4442A41AFAFCCE
2,SRX6793620,SRP220133,NextSeq 500,SRS5335553,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057698,Flank somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,583K,"SAMN12684361,GSM4057698",,,,,,,,,,SAC,2026-03-10,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059713,B18481D02FAC8DCE753FDE79BF07BA5D
3,SRX6793619,SRP220133,NextSeq 500,SRS5335552,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057697,Hindlimb somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,710HL,"SAMN12684345,GSM4057697",,,,,,,,,,SAC,2026-03-10,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059712,1B57FA31B14B8B642C9E27A2C2001877
4,SRX6793618,SRP220133,NextSeq 500,SRS5335551,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057696,Hindlimb somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,707HL,"SAMN12684346,GSM4057696",,,,,,,,,,SAC,2026-03-10,RNA was harvested via lysis in Trizol followed by isolation using RNeasy micro columns. Libraries were synthesized from 200ng total RNA using the stranded TruSeq kit (Illumina) followed by 15 cycles of amplification.,,,,Embryonic tissue,,HH18,,TRANSCRIPTOMIC,cDNA,SRR10059711,F1977DDB98252A0DFE448A807A193AED
5,SRX6793617,SRP220133,NextSeq 500,SRS5335550,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057695,Hi

#### comments

In [12]:
library.loc[:,'comment'] = 'PMID: 31668620'

#### save complete file with correct columns

In [13]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX6793622,SRP220133,NextSeq 500,SRS5335555,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057700,Flank somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,707K,"SAMN12684357,GSM4057700",,,,,,,PMID: 31668620,,,SAC,2026-03-10
1,SRX6793621,SRP220133,NextSeq 500,SRS5335554,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057699,Flank somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,595K,"SAMN12684360,GSM4057699",,,,,,,PMID: 31668620,,,SAC,2026-03-10
2,SRX6793620,SRP220133,NextSeq 500,SRS5335553,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057698,Flank somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,583K,"SAMN12684361,GSM4057698",,,,,,,PMID: 31668620,,,SAC,2026-03-10
3,SRX6793619,SRP220133,NextSeq 500,SRS5335552,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057697,Hindlimb somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,710HL,"SAMN12684345,GSM4057697",,,,,,,PMID: 31668620,,,SAC,2026-03-10
4,SRX6793618,SRP220133,NextSeq 500,SRS5335551,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057696,Hindlimb somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,707HL,"SAMN12684346,GSM4057696",,,,,,,PMID: 31668620,,,SAC,2026-03-10
5,SRX6793617,SRP220133,NextSeq 500,SRS5335550,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057695,Hindlimb somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,595HL,"SAMN12684347,GSM4057695",,,,,,,PMID: 31668620,,,SAC,2026-03-10
6,SRX6793616,SRP220133,NextSeq 500,SRS5335549,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057694,Hindlimb somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,583HL,"SAMN12684348,GSM4057694",,,,,,,PMID: 31668620,,,SAC,2026-03-10
7,SRX6793615,SRP220133,NextSeq 500,SRS5335548,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057693,Forelimb somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,710FL,"SAMN12684352,GSM4057693",,,,,,,PMID: 31668620,,,SAC,2026-03-10
8,SRX6793614,SRP220133,NextSeq 500,SRS5335547,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057692,Forelimb somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,707FL,"SAMN12684353,GSM4057692",,,,,,,PMID: 31668620,,,SAC,2026-03-10
9,SRX6793613,SRP220133,NextSeq 500,SRS5335546,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM4057691,Forelimb somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,595FL,"SAMN12684354,GSM4057691",,,,,,,PMID: 31668620,,,SAC,2026-03-10


### experiment annotations

In [14]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP220133,Attenuated FGF signaling underlies the forelimb heterochrony in the emu Dromaius novaehollandiae (RNA-seq data),"The evolutionary origin of powered flight was fundamental to the establishment and radiation of the avian clade, and it remains a salient feature of modern birds. However, flight has been lost multiple times throughout the evolution of the avian lineage. Ratites (flightless palaeognaths, including the emu, ostrich and other well-known groups) are perhaps the most notable flightless birds, and their convergent losses of flight often coincide with adaptations to a cursorial lifestyle, including robust legs, digit loss, and reduced wings. Although there is a wealth of comparative anatomical knowledge for several ratites, the underlying genetic mechanisms producing these changes remain debated. Here we use a multidisciplinary approach employing embryological, genetic, and genomic techniques to interrogate the mechanisms underlying the delay in forelimb development contributing to the diminution of the forelimb in emu embryos. We show that the epithelial to mesenchymal transition (EMT) in the lateral plate mesoderm (LPM) and muscle precursor migration, the initiating events of limb formation, occur at equivalent stages in the emu and chick. However, the early emu forelimb fails to proliferate at HH18. The unique emu forelimb expression of Nkx2.5, previously associated with diminished wing development, does not initiate until after this stage, concomitant with migration of myoblasts into the limb bud, and hence would not appear to be the proximal cause of limb reduction in this species. In contrast, RNA-sequencing of HH18 limb tissues reveals significantly lower Fgf10 expression in the emu forelimb. Artificially increasing mesenchymal Fgf10 expression to the nascent emu wing induces ectodermal Fgf8 expression and results in a proliferative limb bud. Analyzing open chromatin reveals differentially active regulatory elements near Fgf10 and Sall-1 in the emu wing compared to emu hindlimb and both chicken limbs. Additionally, we show that the Sall-1 enhancer activity is dependent on an Ets transcription factor-binding site likely mediated by Fgf-signaling. Taken together, our results support a model where regulatory changes result in lower expression of Fgf10 and a concomitant failure to induce the genes required for limb proliferation in the early emu wing bud at HH18. Overall design: Comparison of different tissue between the emu and the chick",SRA,,,,,,GSE136774,PRJNA563642,31668620,,10.1016/j.cub.2019.09.014,,


#### experiment and protocol details

In [15]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

40

In [16]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP220133,Attenuated FGF signaling underlies the forelimb heterochrony in the emu Dromaius novaehollandiae (RNA-seq data),"The evolutionary origin of powered flight was fundamental to the establishment and radiation of the avian clade, and it remains a salient feature of modern birds. However, flight has been lost multiple times throughout the evolution of the avian lineage. Ratites (flightless palaeognaths, including the emu, ostrich and other well-known groups) are perhaps the most notable flightless birds, and their convergent losses of flight often coincide with adaptations to a cursorial lifestyle, including robust legs, digit loss, and reduced wings. Although there is a wealth of comparative anatomical knowledge for several ratites, the underlying genetic mechanisms producing these changes remain debated. Here we use a multidisciplinary approach employing embryological, genetic, and genomic techniques to interrogate the mechanisms underlying the delay in forelimb development contributing to the diminution of the forelimb in emu embryos. We show that the epithelial to mesenchymal transition (EMT) in the lateral plate mesoderm (LPM) and muscle precursor migration, the initiating events of limb formation, occur at equivalent stages in the emu and chick. However, the early emu forelimb fails to proliferate at HH18. The unique emu forelimb expression of Nkx2.5, previously associated with diminished wing development, does not initiate until after this stage, concomitant with migration of myoblasts into the limb bud, and hence would not appear to be the proximal cause of limb reduction in this species. In contrast, RNA-sequencing of HH18 limb tissues reveals significantly lower Fgf10 expression in the emu forelimb. Artificially increasing mesenchymal Fgf10 expression to the nascent emu wing induces ectodermal Fgf8 expression and results in a proliferative limb bud. Analyzing open chromatin reveals differentially active regulatory elements near Fgf10 and Sall-1 in the emu wing compared to emu hindlimb and both chicken limbs. Additionally, we show that the Sall-1 enhancer activity is dependent on an Ets transcription factor-binding site likely mediated by Fgf-signaling. Taken together, our results support a model where regulatory changes result in lower expression of Fgf10 and a concomitant failure to induce the genes required for limb proliferation in the early emu wing bud at HH18. Overall design: Comparison of different tissue between the emu and the chick",SRA,total,Bgee 1K,40,"TruSeq Stranded mRNA, KAPA mRNA HyperPrep Kit",full_length,GSE136774,PRJNA563642,31668620,,10.1016/j.cub.2019.09.014,,


#### paper and xrefs

In [17]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
#experiment.loc[:,'PMID'] = ''
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC6834345/'
#experiment.loc[:,'DOI'] = ''
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP220133,Attenuated FGF signaling underlies the forelimb heterochrony in the emu Dromaius novaehollandiae (RNA-seq data),"The evolutionary origin of powered flight was fundamental to the establishment and radiation of the avian clade, and it remains a salient feature of modern birds. However, flight has been lost multiple times throughout the evolution of the avian lineage. Ratites (flightless palaeognaths, including the emu, ostrich and other well-known groups) are perhaps the most notable flightless birds, and their convergent losses of flight often coincide with adaptations to a cursorial lifestyle, including robust legs, digit loss, and reduced wings. Although there is a wealth of comparative anatomical knowledge for several ratites, the underlying genetic mechanisms producing these changes remain debated. Here we use a multidisciplinary approach employing embryological, genetic, and genomic techniques to interrogate the mechanisms underlying the delay in forelimb development contributing to the diminution of the forelimb in emu embryos. We show that the epithelial to mesenchymal transition (EMT) in the lateral plate mesoderm (LPM) and muscle precursor migration, the initiating events of limb formation, occur at equivalent stages in the emu and chick. However, the early emu forelimb fails to proliferate at HH18. The unique emu forelimb expression of Nkx2.5, previously associated with diminished wing development, does not initiate until after this stage, concomitant with migration of myoblasts into the limb bud, and hence would not appear to be the proximal cause of limb reduction in this species. In contrast, RNA-sequencing of HH18 limb tissues reveals significantly lower Fgf10 expression in the emu forelimb. Artificially increasing mesenchymal Fgf10 expression to the nascent emu wing induces ectodermal Fgf8 expression and results in a proliferative limb bud. Analyzing open chromatin reveals differentially active regulatory elements near Fgf10 and Sall-1 in the emu wing compared to emu hindlimb and both chicken limbs. Additionally, we show that the Sall-1 enhancer activity is dependent on an Ets transcription factor-binding site likely mediated by Fgf-signaling. Taken together, our results support a model where regulatory changes result in lower expression of Fgf10 and a concomitant failure to induce the genes required for limb proliferation in the early emu wing bud at HH18. Overall design: Comparison of different tissue between the emu and the chick",SRA,total,Bgee 1K,40,"TruSeq Stranded mRNA, KAPA mRNA HyperPrep Kit",full_length,GSE136774,PRJNA563642,31668620,https://pmc.ncbi.nlm.nih.gov/articles/PMC6834345/,10.1016/j.cub.2019.09.014,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [18]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [19]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

#### to add things here

#### check columns match

In [22]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [23]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
55818,SRX8745359,SRP272247,HiSeq X Ten,SRS7017396,UBERON:0000468,multicellular organism,UBERON:0000112,sexually immature stage,,"liver, pancreas, stomach (proventriculus) and ...",six months old,other,not documented,other,NA,,,8790,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,Model organism or animal sample from Dromaius ...,SAMN15558384,6,month,,,,,"PMID: 33986452, PMID: 30963837",,,SAC,2026-03-10
55819,SRX8745358,SRP272247,HiSeq X Ten,SRS7017395,UBERON:0000468,multicellular organism,UBERON:0000112,sexually immature stage,,"liver, pancreas, stomach (proventriculus) and ...",three months old,other,not documented,other,NA,,,8801,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,Model organism or animal sample from Struthio ...,SAMN15558383,3,month,,,,,"PMID: 33986452, PMID: 30963837",,,SAC,2026-03-10
55820,SRX6793622,SRP220133,NextSeq 500,SRS5335555,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Flank somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,707K,"SAMN12684357,GSM4057700",,,,,,,PMID: 31668620,,,SAC,2026-03-10
55821,SRX6793621,SRP220133,NextSeq 500,SRS5335554,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Flank somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,595K,"SAMN12684360,GSM4057699",,,,,,,PMID: 31668620,,,SAC,2026-03-10
55822,SRX6793620,SRP220133,NextSeq 500,SRS5335553,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Flank somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,583K,"SAMN12684361,GSM4057698",,,,,,,PMID: 31668620,,,SAC,2026-03-10
55823,SRX6793619,SRP220133,NextSeq 500,SRS5335552,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Hindlimb somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,710HL,"SAMN12684345,GSM4057697",,,,,,,PMID: 31668620,,,SAC,2026-03-10
55824,SRX6793618,SRP220133,NextSeq 500,SRS5335551,UBERON:0004874,somatopleure,UBERON:0000111,organogenesis stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Hindlimb somatopleure,HH18,missing child term,not documented,other,NA,,,8790,TruSeq Stranded mRNA,full_length,polyA,,,707HL,"SAMN12684346,GSM4057696",,,,,,,PMID: 31668620,,,SAC,2026-03-10


In [24]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1085,SRP336229,Seasonal and sex- dependent gene expression in...,Emus (Dromaius novaehollandiae) are birds farm...,SRA,total,Bgee 1K,24,TruSeq RNA Sample Preparation Kit,full_length,,PRJNA761725,35676317,https://pmc.ncbi.nlm.nih.gov/articles/PMC9177602/,10.1038/s41598-022-13681-5,,
1086,SRP272247,Diet of ancestral birds,This project aims to examine the adaptive evol...,SRA,total,Bgee 1K,2,NEBNext Ultra RNA Library Prep Kit,full_length,,PRJNA646721,33986452,https://pmc.ncbi.nlm.nih.gov/articles/PMC8119460/,10.1038/s42003-021-02067-4,30963837[PMID],
1087,SRP220133,Attenuated FGF signaling underlies the forelim...,The evolutionary origin of powered flight was ...,SRA,total,Bgee 1K,40,"TruSeq Stranded mRNA, KAPA mRNA HyperPrep Kit",full_length,GSE136774,PRJNA563642,31668620,https://pmc.ncbi.nlm.nih.gov/articles/PMC6834345/,10.1016/j.cub.2019.09.014,,


### add annotations to git

In [25]:
! git pull

Already up to date.


In [26]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [27]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../NCBI_output/
	./
	../SRP240625/

no changes added to commit (use "git add" and/or "git commit -a")


In [28]:
! git add $git_experiment_path $git_library_path

In [29]:
! git commit -m $commit_message_exp

[develop c82c871] adding annotated bulk experiment SRP220133
 2 files changed, 41 insertions(+)


In [30]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 4.17 KiB | 1.04 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   98031be..c82c871  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push